Tests per verificare come vengono salvati gli excel nelle varie modifiche e cicli di pulizia

# Import e set up
da "adni_cleaning1"

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd

dataCleaner = DataCleaner(support_file_path='prova_0.xlsx')
client = DatalakeClient()

# Download the raw files

In [2]:
search = client.query_files(
    query={'custom.level' : 'raw', 'custom.source' : 'ADNI'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

# Support file managment
operazione per popolare il file support file per i file considerati

In [3]:
support_file_path = 'prova_0'
support_file = pd.read_excel(support_file_path+'.xlsx')
for file_name in zip_files.keys():
    df = zip_files[file_name]
    infoSupportFile = InfoSupportFile(support_file, df, file_name)
    # delate the rows of the support file that are not in the df
    support_file, file_code = infoSupportFile.filter_variables()
    print(file_code)
    if file_code not in support_file['file_code'].unique().tolist():
        print('file_code {} not in support_file'.format(file_code))
        continue
    # find the population variable code, and if not in support_file, add it
    pop, support_file = infoSupportFile.find_population_variable()
    # get the variable info and add it to the support_file    
    for key in df.keys():
        if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
            support_file = infoSupportFile.get_varible_info(key)
# save the updated support file
save_df(df_to_save=support_file, output_path=support_file_path)

create_new_support_file(support_file, support_file_path, new_name='prova_1')

file_code 'NEUROPATH' non trovato nel support_file.
NEUROPATH
file_code NEUROPATH not in support_file
ADNIMERGE
ADSP_PHC_BIOMARKER
BLCHANGE
DXSUM
MMSE
PTDEMOG


# Cleaning 1 - file specific 
## ADNIMERGE

In [4]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [5]:
file_name


'ADNIMERGE_25Jul2025.csv'

In [6]:
df_new = dataset.copy(deep=True) 

In [7]:
# Important columns
columns_must_be_verified = ['APOE4', 'MMSE', 'Ventricles', 'Hippocampus', 'AGE']
single_column_required = ['DX']

In [8]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
processed_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
processed_df = dataCleaner.drop_if_all_none(processed_df, single_column_required)
processed_df.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
processed_df['VISCODE'] = processed_df['VISCODE'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [9]:
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
processed_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DX')
processed_df = dataCleaner.to_date_format(processed_df, ['EXAMDATE'])
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0],['AGE_bl'],'raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE


In [10]:
new_support_file_path = 'prova_1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
df_cleaned1 = final_df.copy(deep=True)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [11]:
df_cleaned1.keys()

Index(['PTID', 'RID', 'COLPROT', 'VISCODE', 'EXAMDATE', 'AGE', 'PTGENDER',
       'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'PIB', 'AV45',
       'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'MOCA',
       'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV',
       'DX', 'AGE_bl'],
      dtype='object')

In [12]:
infoSupportFile = InfoSupportFile(new_support_file, df_cleaned1, file_name)
   
for key in df_cleaned1.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

# VERIFY - cleaning 1 process

In [13]:
excel = new_support_file[new_support_file['file_code'] == 'ADNIMERGE']['variable_code'].tolist()
dataset = df_cleaned1.keys().tolist()
excel_not_in_dataset = [x for x in excel if x not in dataset]
dataset_not_in_excel = [x for x in dataset if x not in excel]
print('excel_not_in_dataset: ', excel_not_in_dataset)
print('dataset_not_in_excel: ', dataset_not_in_excel)
print('dataset keys: ', dataset)

excel_not_in_dataset:  []
dataset_not_in_excel:  []
dataset keys:  ['PTID', 'RID', 'COLPROT', 'VISCODE', 'EXAMDATE', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'PIB', 'AV45', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'MOCA', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'DX', 'AGE_bl']


# Cleaning 2

In [14]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'prova_1'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'prova_2'
create_new_support_file(support_file, support_file_path, new_name=new_name, rename=False)
dataCleaner = DataCleaner(support_file_path=new_name+'.xlsx')

In [15]:
file_name = 'ADNIMERGE_25Jul2025_1.csv'
df_new2 = df_cleaned1.copy(deep=True)
df_cleaned, file_code, temp_support_file = dataCleaner.remove_param_few_subjects(df_new2, file_name, prefix='cleaned/single_file')
# --> funzione che trova i soggetti che hanno solo una visita quindi elimina quelle righe


# Eliminare soggetti con solo 1 visita
df_cleaned= dataCleaner.remove_sub_1visit(df_cleaned)
# --> funzione che trova i parametri identificati da eliminare  ==> eliminare le colonne dal df

final_df, bool_var = dataCleaner.classes_to_dummies(df_cleaned, col_list=['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']) #, 'DX'])

# Nuovi metadati (cofattori/fattori, scal/intervallo)
updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_2')

new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_02')
print(new_file_name)

# aggiunta di righe per i nuovi parametri
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)

# aggiornamento metadati nel support file
new_support_file = dataCleaner.update_metadati_support(new_support_file)
# get info into the new support file
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, 'prova_2')

ADNIMERGE_25Jul2025_02.csv
var:  AGE colonna:  cofattori
idx:  Index([5], dtype='int64')
in if not idx.empty
var:  PTGENDER colonna:  cofattori
idx:  Index([6], dtype='int64')
in if not idx.empty
var:  PTEDUCAT colonna:  cofattori
idx:  Index([7], dtype='int64')
in if not idx.empty
var:  PTETHCAT colonna:  cofattori
idx:  Index([8], dtype='int64')
in if not idx.empty
var:  PTRACCAT colonna:  cofattori
idx:  Index([9], dtype='int64')
in if not idx.empty
var:  PTMARRY colonna:  cofattori
idx:  Index([10], dtype='int64')
in if not idx.empty
var:  APOE4 colonna:  cofattori
idx:  Index([11], dtype='int64')
in if not idx.empty
var:  CDRSB colonna:  predittori
idx:  Index([12], dtype='int64')
in if not idx.empty
var:  ADAS11 colonna:  predittori
idx:  Index([13], dtype='int64')
in if not idx.empty
var:  ADAS13 colonna:  predittori
idx:  Index([14], dtype='int64')
in if not idx.empty
var:  MMSE colonna:  predittori
idx:  Index([15], dtype='int64')
in if not idx.empty
var:  RAVLT_immediate colo

In [16]:
print(updated_metadata)

{'file_code': 'ADNIMERGE', 'level': 'cleaned_2', 'population': ['ADNI1', 'ADNIGO', 'ADNI2', 'ADNI3'], 'source': 'ADNI', 'cofattori': ['AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4'], 'predittori': ['CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'DX/CN', 'DX/Dementia', 'DX/MCI'], 'norm_scala': ['CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ'], 'norm_intervallo': ['APOE4', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV']}


In [17]:
excel = new_support_file[new_support_file['file_code'] == 'ADNIMERGE']['variable_code'].tolist()
dataset = final_df.keys().tolist()
excel_not_in_dataset = [x for x in excel if x not in dataset]
dataset_not_in_excel = [x for x in dataset if x not in excel]
print('excel_not_in_dataset: ', excel_not_in_dataset)
print('dataset_not_in_excel: ', dataset_not_in_excel)
print('dataset keys: ', dataset)

excel_not_in_dataset:  []
dataset_not_in_excel:  []
dataset keys:  ['PTID', 'RID', 'COLPROT', 'VISCODE', 'EXAMDATE', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'AGE_bl', 'DX/CN', 'DX/Dementia', 'DX/MCI']


In [18]:
dataset1 = df_cleaned1.keys().tolist()
dataset2 = final_df.keys().tolist()
dataset1_not_in_dataset2 = [x for x in dataset1 if x not in dataset2]
dataset2_not_in_dataset1 = [x for x in dataset2 if x not in dataset1]
print('dataset1_not_in_dataset2: ', dataset1_not_in_dataset2)
print('dataset2_not_in_dataset1: ', dataset2_not_in_dataset1)
print('dataset1: ', dataset1)
print('dataset2: ', dataset2)

dataset1_not_in_dataset2:  ['PIB', 'AV45', 'MOCA', 'DX']
dataset2_not_in_dataset1:  ['DX/CN', 'DX/Dementia', 'DX/MCI']
dataset1:  ['PTID', 'RID', 'COLPROT', 'VISCODE', 'EXAMDATE', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'PIB', 'AV45', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'MOCA', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'DX', 'AGE_bl']
dataset2:  ['PTID', 'RID', 'COLPROT', 'VISCODE', 'EXAMDATE', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'AGE_bl', 'DX/CN', 'DX/Dementia', 'DX/MCI']
